# Projeto 05 — Detecção de Intrusão em Redes com MLP
**Disciplina:** PAM0466 – Sistemas Inteligentes  
**Docente:** Pedro Thiago Valério de Souza  
**Semestre:** 2026.1

---

## Bibliotecas Utilizadas

Importação de todas as dependências necessárias para o projeto:

- **pandas / numpy** — manipulação e processamento dos dados
- **torch / torch.nn** — construção e treinamento da rede neural MLP
- **sklearn** — normalização, métricas e matriz de confusão
- **matplotlib** — visualização das curvas de loss e matriz de confusão

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

print("Bibliotecas importadas com sucesso!")
print(f"PyTorch versão: {torch.__version__}")

---

## Atividade 1 — Preparação dos Dados

### 1.1 Carregamento dos Arquivos

O dataset NSL-KDD é carregado a partir dos arquivos `KDDTrain+.txt` e `KDDTest+.txt`.  
Como os arquivos não possuem cabeçalho, os 43 nomes de colunas são atribuídos manualmente  
conforme especificado no enunciado do projeto.

| Arquivo | Amostras |
|---|---|
| KDDTrain+.txt | 125.973 |
| KDDTest+.txt | 22.544 |

Cada registro representa uma conexão de rede com **41 atributos** divididos em:
- **Básicos** — duração, protocolo, serviço, flag, bytes transferidos
- **Conteúdo** — tentativas de login, acesso a arquivos sensíveis, comandos shell
- **Tráfego** — estatísticas das últimas conexões ao mesmo host/serviço

In [ ]:
PATH = "/kaggle/input/datasets/hassan06/nslkdd/"

col_names = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "attack_type", "difficulty_level"
]

df_train = pd.read_csv(PATH + "KDDTrain+.txt", header=None, names=col_names)
df_test  = pd.read_csv(PATH + "KDDTest+.txt",  header=None, names=col_names)

print(f"Treino:  {df_train.shape}")
print(f"Teste:   {df_test.shape}")
df_train.head(3)

### 1.2 Limpeza e Binarização do Rótulo

O rótulo original possui 23 tipos de ataque agrupados em 4 categorias  
(DoS, Probe, R2L, U2R) além da classe `normal`.  

Neste projeto o problema é tratado como **classificação binária**:

| Rótulo original | Rótulo binário |
|---|---|
| `normal` | 0 |
| qualquer ataque | 1 |

A coluna `difficulty_level` é descartada por não ser relevante para o modelo.

In [ ]:
# Remove a coluna difficulty_level (não usaremos)
df_train.drop(columns=["difficulty_level"], inplace=True)
df_test.drop(columns=["difficulty_level"],  inplace=True)

# Binariza o rótulo: normal → 0, qualquer ataque → 1
df_train["label"] = (df_train["attack_type"] != "normal").astype(int)
df_test["label"]  = (df_test["attack_type"]  != "normal").astype(int)

# Remove a coluna attack_type (já extraímos o rótulo)
df_train.drop(columns=["attack_type"], inplace=True)
df_test.drop(columns=["attack_type"],  inplace=True)

# Confere a distribuição das classes
print("Distribuição no TREINO:")
print(df_train["label"].value_counts())
print(f"\nDistribuição no TESTE:")
print(df_test["label"].value_counts())

### 1.3 One-Hot Encoding

Três atributos são **categóricos** (texto) e precisam ser convertidos para formato numérico:

| Coluna | Exemplo de valores |
|---|---|
| `protocol_type` | tcp, udp, icmp |
| `service` | http, ftp, smtp, ... |
| `flag` | SF, S0, REJ, ... |

O `pd.get_dummies` transforma cada categoria em uma coluna binária separada.  
Em seguida, `align` garante que treino e teste tenham **exatamente as mesmas colunas**,  
preenchendo com 0 categorias ausentes no teste.

In [ ]:
# Colunas categóricas que precisam virar números
cat_cols = ["protocol_type", "service", "flag"]

# Aplica one-hot encoding
df_train = pd.get_dummies(df_train, columns=cat_cols)
df_test  = pd.get_dummies(df_test,  columns=cat_cols)

# Alinha as colunas: teste pode ter categorias que não aparecem no treino
df_train, df_test = df_train.align(df_test, join="left", axis=1, fill_value=0)

print(f"Colunas após encoding — Treino: {df_train.shape[1]}")
print(f"Colunas após encoding — Teste:  {df_test.shape[1]}")
print(f"\nExemplo de colunas geradas:")
print([c for c in df_train.columns if c.startswith("protocol") or c.startswith("flag") or c.startswith("service")][:10])

### 1.4 Separação de Features e Rótulos

As colunas de entrada (features) são separadas da coluna de saída (label).  
Todos os valores são convertidos para `float32`, formato exigido pelo PyTorch.

In [ ]:
# Todas as colunas exceto o rótulo são features
feature_cols = [c for c in df_train.columns if c != "label"]

X_train_full = df_train[feature_cols].values.astype(np.float32)
y_train_full = df_train["label"].values.astype(np.float32)

X_test = df_test[feature_cols].values.astype(np.float32)
y_test = df_test["label"].values.astype(np.float32)

print(f"X_train_full: {X_train_full.shape}")
print(f"y_train_full: {y_train_full.shape}")
print(f"X_test:       {X_test.shape}")
print(f"y_test:       {y_test.shape}")

### 1.5 Divisão Treino/Validação e Normalização Z-score

O conjunto de treino é dividido em **80% treino e 20% validação**,  
**sem embaralhamento**, conforme exigido pelo enunciado.

A normalização Z-score é aplicada em todos os atributos numéricos:

$$x' = \frac{x - \mu}{\sigma}$$

> ⚠️ As estatísticas μ e σ são calculadas **apenas sobre o conjunto de treino**  
> e aplicadas ao conjunto de validação e teste.  
> Isso evita *data leakage* — o modelo nunca "vê" informações dos dados futuros.

In [ ]:
# Split 80/20 sem embaralhamento
split = int(0.8 * len(X_train_full))

X_train, X_val = X_train_full[:split], X_train_full[split:]
y_train, y_val = y_train_full[:split], y_train_full[split:]

# Normalização Z-score — fit APENAS no treino
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Treino:    {X_train.shape[0]} amostras")
print(f"Validação: {X_val.shape[0]} amostras")
print(f"Teste:     {X_test.shape[0]} amostras")
print(f"\nMédia de uma feature antes da norm: {X_train_full[0][0]:.4f}")
print(f"Média de uma feature após a norm:  {X_train[0][0]:.4f}")

---

## Atividade 2 — Implementação da MLP em PyTorch

### 2.1 Arquitetura da Rede

A classe `MLP` herda de `nn.Module` e é construída de forma **flexível**,  
aceitando qualquer lista de camadas ocultas via parâmetro `hidden_dims`.

Cada camada oculta é composta por 4 elementos em sequência:

| Componente | Função |
|---|---|
| `nn.Linear` | Transformação linear (pesos e bias) |
| `nn.BatchNorm1d` | Estabiliza o treinamento normalizando as ativações |
| `nn.ReLU` | Introduz não-linearidade: `f(x) = max(0, x)` |
| `nn.Dropout(0.3)` | Desliga 30% dos neurônios aleatoriamente — evita overfitting |

> ⚠️ A camada de saída possui **apenas** `nn.Linear`, sem função de ativação.  
> Isso é obrigatório pois `BCEWithLogitsLoss` já aplica a sigmoid internamente,  
> de forma numericamente mais estável.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout_rate=0.3):
        super(MLP, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))   # conexão entre camadas
            layers.append(nn.BatchNorm1d(h_dim))         # normalização interna
            layers.append(nn.ReLU())                     # ativação não-linear
            layers.append(nn.Dropout(dropout_rate))      # regularização
            prev_dim = h_dim
        
        # Camada de saída — SEM ativação
        layers.append(nn.Linear(prev_dim, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x).squeeze(1)

# Testa a arquitetura com uma topologia exemplo
modelo_teste = MLP(input_dim=X_train.shape[1], hidden_dims=[256, 128, 64])
print(modelo_teste)
print(f"\nTotal de parâmetros: {sum(p.numel() for p in modelo_teste.parameters()):,}")

### 2.2 Função de Treinamento

O loop de treinamento segue o ciclo padrão do PyTorch para cada mini-batch:

In [ ]:
def treinar(X_tr, y_tr, X_vl, y_vl, hidden_dims,
            lr=1e-3, epochs=50, batch_size=256,
            dropout=0.3, patience=10):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Usando: {device}")

    # Converte para tensores PyTorch
    Xt = torch.tensor(X_tr, dtype=torch.float32).to(device)
    yt = torch.tensor(y_tr, dtype=torch.float32).to(device)
    Xv = torch.tensor(X_vl, dtype=torch.float32).to(device)
    yv = torch.tensor(y_vl, dtype=torch.float32).to(device)

    # DataLoader com mini-batches de 256
    dataset = TensorDataset(Xt, yt)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Instancia modelo, loss e otimizador
    model     = MLP(X_tr.shape[1], hidden_dims, dropout).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []
    best_val_loss    = float("inf")
    best_state       = None
    epochs_no_improv = 0

    for epoch in range(1, epochs + 1):

        # ── Treino ──
        model.train()
        epoch_loss = 0.0
        for Xb, yb in loader:
            optimizer.zero_grad()        # zera gradientes anteriores
            logits = model(Xb)           # forward pass
            loss   = criterion(logits, yb)  # calcula loss
            loss.backward()              # backpropagation
            optimizer.step()             # atualiza pesos
            epoch_loss += loss.item() * len(yb)

        train_loss = epoch_loss / len(y_tr)

        # ── Validação ──
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(Xv), yv).item()

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # ── Early stopping ──
        if val_loss < best_val_loss - 1e-4:
            best_val_loss    = val_loss
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improv = 0
        else:
            epochs_no_improv += 1

        if epoch % 10 == 0:
            print(f"  Época {epoch:3d}/{epochs} | "
                  f"Loss treino: {train_loss:.4f} | "
                  f"Loss val: {val_loss:.4f}")

        if epochs_no_improv >= patience:
            print(f"\n  ⚠ Parada antecipada na época {epoch}!")
            break

    # Restaura o melhor estado encontrado
    if best_state:
        model.load_state_dict(best_state)

    return model, train_losses, val_losses, device

---

## Atividade 3 — Treinamento e Monitoramento

### 3.1 Busca de Topologia

Quatro topologias são avaliadas para identificar a de melhor desempenho na validação:

| Topologia | Camadas ocultas |
|---|---|
| 128-64 | 2 camadas |
| 256-128-64 | 3 camadas |
| **512-256-128** | **3 camadas — melhor resultado** |
| 256-128-64-32 | 4 camadas |

A escolha é baseada na **menor loss de validação** ao final do treinamento.

In [ ]:
topologias = {
    "128-64":        [128, 64],
    "256-128-64":    [256, 128, 64],
    "512-256-128":   [512, 256, 128],
    "256-128-64-32": [256, 128, 64, 32],
}

resultados = {}
melhor_nome     = None
melhor_val_loss = float("inf")
melhor_modelo   = None
melhor_device   = None

for nome, dims in topologias.items():
    print(f"\n{'='*50}")
    print(f"Topologia: {nome}")
    print(f"{'='*50}")
    
    modelo, train_hist, val_hist, device = treinar(
        X_train, y_train, X_val, y_val,
        hidden_dims=dims, epochs=50, patience=10
    )
    
    resultados[nome] = (train_hist, val_hist)
    final_val = val_hist[-1]
    print(f"\n  → Loss validação final: {final_val:.4f}")
    
    if final_val < melhor_val_loss:
        melhor_val_loss = final_val
        melhor_nome     = nome
        melhor_modelo   = modelo
        melhor_device   = device

print(f"\n{'='*50}")
print(f"✔ Melhor topologia: {melhor_nome}")
print(f"✔ Loss validação:   {melhor_val_loss:.4f}")

### 3.2 Curvas de Loss — Todas as Topologias

Comparação visual do comportamento de treino e validação para cada arquitetura testada.  
Curvas próximas e descendentes indicam bom aprendizado sem overfitting.

In [ ]:
# Curvas da melhor topologia na mesma figura (conforme enunciado)
train_hist, val_hist = resultados[melhor_nome]

plt.figure(figsize=(9, 5))
plt.plot(train_hist, label="Loss Treino",    linewidth=2)
plt.plot(val_hist,   label="Loss Validação", linewidth=2, linestyle="--")
plt.xlabel("Épocas")
plt.ylabel("BCE Loss")
plt.title(f"Curvas de Loss — Melhor Topologia ({melhor_nome})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("loss_melhor_topologia.png", dpi=150)
plt.show()

---

## Atividade 4 — Avaliação Final

### 4.1 Predição no Conjunto de Teste

O melhor modelo (512-256-128) é avaliado no `KDDTest+.txt` —  
dados que **nunca foram vistos** durante o treinamento ou validação.

A saída da rede (logit) é convertida em probabilidade pela sigmoid:

$$p = \frac{1}{1 + e^{-z}}$$

O limiar de **0.5** é aplicado para obter os rótulos preditos:

$$\hat{y} = \begin{cases} 1 & \text{se } p \geq 0.5 \\ 0 & \text{se } p < 0.5 \end{cases}$$

In [ ]:
# Converte o conjunto de teste para tensor
Xte = torch.tensor(X_test, dtype=torch.float32).to(melhor_device)

# Forward pass sem gradiente
melhor_modelo.eval()
with torch.no_grad():
    logits = melhor_modelo(Xte).cpu().numpy()

# Aplica sigmoid e threshold de 0.5
probs  = 1 / (1 + np.exp(-logits))
y_pred = (probs >= 0.5).astype(int)

# Métricas
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print("=" * 40)
print("RESULTADOS NO CONJUNTO DE TESTE")
print("=" * 40)
print(f"Acurácia:  {acc:.4f}  ({acc*100:.2f}%)")
print(f"Precisão:  {prec:.4f}  ({prec*100:.2f}%)")
print(f"Recall:    {rec:.4f}  ({rec*100:.2f}%)")
print(f"F1-Score:  {f1:.4f}  ({f1*100:.2f}%)")

### 4.2 Matriz de Confusão

|  | Previsto Normal | Previsto Ataque |
|---|---|---|
| **Real Normal** | Verdadeiro Negativo (VN) | Falso Positivo (FP) |
| **Real Ataque** | Falso Negativo (FN) ⚠️ | Verdadeiro Positivo (VP) |

> ⚠️ O **Falso Negativo** é o erro mais crítico em um IDS:  
> um ataque classificado como normal passa despercebido sem nenhum alerta.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=["Normal (0)", "Ataque (1)"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")

ax.set_title("Matriz de Confusão — Conjunto de Teste", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# Detalha os erros
VN, FP, FN, VP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
print(f"Verdadeiros Negativos (normal correto):  {VN:>6}")
print(f"Falsos Positivos      (normal → ataque): {FP:>6}")
print(f"Falsos Negativos      (ataque → normal): {FN:>6}  ← crítico!")
print(f"Verdadeiros Positivos (ataque correto):  {VP:>6}")

### 4.3 Impacto do Limiar de Decisão

Em um IDS em produção, reduzir o threshold aumenta o recall  
(menos ataques passam despercebidos) ao custo de mais falsos alarmes.

**O custo de um ataque não detectado supera o custo de investigar um alarme falso.**  
Por isso, em segurança recomenda-se priorizar **recall sobre precisão**,  
utilizando thresholds mais baixos (0.2 a 0.3).

In [ ]:
thresholds = [0.5, 0.4, 0.3, 0.2]

print(f"{'Threshold':>10} | {'Acurácia':>9} | {'Precisão':>9} | {'Recall':>9} | {'F1':>9} | {'FN':>7} | {'FP':>7}")
print("-" * 75)

for t in thresholds:
    y_pred_t = (probs >= t).astype(int)
    print(f"{t:>10.1f} | "
          f"{accuracy_score(y_test, y_pred_t):>9.4f} | "
          f"{precision_score(y_test, y_pred_t):>9.4f} | "
          f"{recall_score(y_test, y_pred_t):>9.4f} | "
          f"{f1_score(y_test, y_pred_t):>9.4f} | "
          f"{confusion_matrix(y_test, y_pred_t)[1,0]:>7} | "
          f"{confusion_matrix(y_test, y_pred_t)[0,1]:>7}")